# Right-Size It

**Find the cheapest model that is still good enough for your task.** Run one prompt across a few models with your own key, read the answers, mark which are good enough, and the tool tells you the cheapest of the ones *you* accepted, and how much it saves. On many everyday tasks a cheap model matches a flagship that costs a hundred times as much. About a minute, free, no signup.

**You make the quality call. The tool only adds up cost.** It never decides which answer is good enough, that is yours. This notebook is **pre-filled with a demonstration default** (a simple task across a cheap, a mid, and a flagship model, with an example acceptable selection), so your first run already shows the point.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API keys and calls the providers directly. **Your keys never leave your machine / your Colab account and are never sent to pysolvr.** You are responsible for your own usage and costs. You decide which answers are good enough; the tool only adds up cost. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O right_size_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/right-size/right_size_core.py
!wget -q -O prices.json        https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/right-size/prices.json
print('ready')

## 2. Add your API keys (do NOT type them into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store them *outside* the notebook instead. The cell below only *reads* them.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for each provider you need, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for each secret. You need a key for each provider whose model you include.

### On local Jupyter, environment variables
Set them in your shell *before* launching Jupyter, then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your keys stay on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Pick your models and prompt, then run them

The cell below is pre-filled with a demonstration: a simple classify task run across a cheap model, a mid one, and a flagship. Run it as-is to see each answer with its cost, numbered.

Use the provider's own model-ID strings (the tool is model-agnostic). To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To right-size **your own** task: set `MODELS` (2 to 6) and `PROMPT` to yours.

In [ ]:
MODELS = ['gpt-4o-mini', 'gpt-4o', 'claude-opus-5-5']
PROMPT = ('Classify this customer message as one of: billing, technical, other. '
          'Reply with only the single word.\n\nMessage: "My card was charged twice this month."')

from right_size_core import right_size_core, format_runs, cheapest_of_acceptable, format_decision

result = right_size_core(prompt=PROMPT, models=MODELS)
print(format_runs(result))

## 4. Mark the good-enough answers, then see the cheapest

Read the answers above and decide which ones are **good enough for your task**. The tool does not judge quality, you do. Put the numbers of the acceptable answers in `ACCEPTABLE`, then run the cell.

The demonstration pre-marks all three as an **example** (a simple classify task even a cheap model handles). For your own task, set `ACCEPTABLE` to the ones *you* judge good enough.

In [ ]:
# The run numbers (from the list above) whose answers YOU judge good enough. Example: [1, 2, 3].
ACCEPTABLE = [1, 2, 3]

decision = cheapest_of_acceptable(result, ACCEPTABLE)
print(format_decision(result, decision))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). You did this by hand for a few models and one prompt. To have a judge score which models clear your quality bar and pick the cheapest automatically, across your whole workload, see [Features](https://llmprover.pysolvr.com/features/?tool=right-size).